In [1]:
import torch
import torch.nn as nn
import re
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import T5Tokenizer, T5EncoderModel

# loading data


/home/jvlk/miniconda3/envs/protbert/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Základní encoder
tokenizer = T5Tokenizer.from_pretrained(
    'Rostlab/prot_t5_xl_half_uniref50-enc',
    do_lower_case=False
)

base_model = T5EncoderModel.from_pretrained(
    "Rostlab/prot_t5_xl_half_uniref50-enc"
)

if device.type == "cpu":
    base_model.to(torch.float32)

def nll_loss(mu, sigma, y):
    return torch.mean(0.5 * ((y - mu) ** 2) / (sigma ** 2) + torch.log(sigma))


class ProteinStabilityUncertaintyModel(nn.Module):
    def __init__(self, encoder, hidden_size=1024):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 2)  # predikujeme [mu, log_sigma]
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state.mean(dim=1)
        pred = self.head(pooled)
        mu = pred[:, 0]  # průměr predikovaného rozdělení
        log_sigma = pred[:, 1]
        sigma = F.softplus(log_sigma) + 1e-6  # zajistíme σ > 0
        return mu, sigma

class ProteinDataset(Dataset):
    def __init__(self, dataframe, tokenizer, target_col="normalized_fitness"):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.target = dataframe[target_col].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        seq = self.df["aa_seq"].iloc[idx]
        # nahradíme neznámé AMK a vložíme mezery
        seq = " ".join(list(re.sub(r"[UZOB]", "X", seq)))
        enc = self.tokenizer(seq, padding="max_length", max_length=512, truncation=True, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["target"] = torch.tensor(self.target[idx], dtype=torch.float32)
        return item

df_train = pd.read_pickle("lehner_dataset.pkl")

train_dataset = ProteinDataset(df_train, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=12, shuffle=True)

# --- Model & optimizer ---
model = ProteinStabilityUncertaintyModel(base_model).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
from tqdm import tqdm

for epoch in range(5):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["target"].to(device)

        mu, sigma = model(input_ids, attention_mask)
        loss = nll_loss(mu, sigma, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item(), avg_loss=total_loss/len(loop))

    print(f"Epoch {epoch+1} finished. Avg Loss: {total_loss/len(train_loader):.4f}")


Epoch 1:   0%|          | 0/50 [00:00<?, ?it/s]